In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
doc_path = "../Retrieval-Augmented-Generation.pdf"

In [4]:
from langchain_community.document_loaders import PyPDFLoader

In [5]:
loader = PyPDFLoader(doc_path)

In [6]:
docs = loader.load()

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [9]:
chunks = splitter.split_documents(docs)
len(chunks)

158

In [10]:
from langchain.vectorstores import Chroma

In [11]:
vectorstore = Chroma.from_documents(chunks, embeddings)

In [12]:
vectorstore_retreiver = vectorstore.as_retriever(search_kwargs={"k": 3})

In [13]:
vectorstore_retreiver

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000023C24F23B00>, search_kwargs={'k': 3})

In [14]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

In [15]:
keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000023C2522D400>)

In [16]:
keyword_retriever.k

4

In [17]:
keyword_retriever.k =  3

In [18]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[vectorstore_retreiver, keyword_retriever], weights=[0.3, 0.7]
)

ensemble_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000023C24F23B00>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000023C2522D400>, k=3)], weights=[0.3, 0.7])

In [19]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.0
)

In [20]:
ensemble_retriever.invoke('What is Abstractive Question Answering?')

[Document(metadata={'total_pages': 19, 'creator': 'LaTeX with hyperref', 'page_label': '6', 'producer': 'pdfTeX-1.40.21', 'title': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'author': '', 'trapped': '/False', 'source': '../Retrieval-Augmented-Generation.pdf', 'creationdate': '2021-04-13T00:48:38+00:00', 'page': 5, 'keywords': '', 'subject': ''}, page_content='even when the correct answer is not in any retrieved document, achieving 11.8% accuracy in such\ncases for NQ, where an extractive model would score 0%.\n4.2 Abstractive Question Answering\nAs shown in Table 2, RAG-Sequence outperforms BART on Open MS-MARCO NLG by 2.6 Bleu\npoints and 2.6 Rouge-L points. RAG approaches state-of-the-art model performance, which is\nimpressive given that (i) those models access gold passages with speciﬁc information required to'),
 Document(metadata={'total_pages': 19, 'title': '', 'subject':

In [21]:
ensemble_retriever.invoke('what is RAG token?')

[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../Retrieval-Augmented-Generation.pdf', 'total_pages': 19, 'page': 6, 'page_label': '7'}, page_content='MS-\nMARCO\ndeﬁne middle\near\nBART ?The middle ear is the part of the ear between the middle ear and the nose.\nRAG-T The middle ear is the portion of the ear internal to the eardrum.\nRAG-S The middle ear includes the tympanic cavity and the three ossicles.\nwhat currency\nneeded in\nscotland\nBART The currency needed in Scotland is Pound sterling.\nRAG-T Pound is the currency needed in Scotland.\nRAG-S The currency needed in Scotland is the pound sterling.\nJeopardy\nQuestion\nGener\n-ation\nWashington'),
 Document(metad

In [22]:
from langchain.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

In [23]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)

In [24]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{query}"),
    ]
)

In [25]:
from langchain.prompts import PromptTemplate

template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you do not have the relevant information needed to provide a verified answer, don't try to make up an answer.
When providing an answer, aim for clarity and precision. Position yourself as a knowledgeable authority on the topic, but also be mindful to explain the information in a manner that is accessible and comprehensible to those without a technical background.
Always say "Do you have any more questions pertaining to this instrument?" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""

prompt = PromptTemplate.from_template(template)

In [26]:
from langchain.chains.combine_documents import create_stuff_documents_chain

In [27]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
question_answer_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nUse the following pieces of context to answer the question at the end.\nIf you don\'t know the answer, just say that you do not have the relevant information needed to provide a verified answer, don\'t try to make up an answer.\nWhen providing an answer, aim for clarity and precision. Position yourself as a knowledgeable authority on the topic, but also be mindful to explain the information in a manner that is accessible and comprehensible to those without a technical background.\nAlways say "Do you have any more questions pertaining to this instrument?" at the end of the answer.\n{context}\nQuestion: {question}\nHelpful Answer:')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Compl

In [28]:
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ensemble_retriever
)

In [29]:
result1 = hybrid_chain.invoke("what is natural language processing?")
print(result1)

{'query': 'what is natural language processing?', 'result': "I don't know."}


In [30]:
query = "What is Abstractive Question Answering?"
response = hybrid_chain.invoke({"query":query})
response

{'query': 'What is Abstractive Question Answering?',
 'result': 'Abstractive Question Answering refers to a type of question answering that goes beyond simply extracting information from a given text. Instead of pulling direct quotes or passages, it generates answers in a free-form manner, often rephrasing or summarizing the information to provide a more coherent and contextually relevant response. This approach allows for more flexibility and creativity in answering questions, as it can synthesize information from multiple sources or present it in a way that is not strictly tied to the original text.'}

In [31]:
response['result']

'Abstractive Question Answering refers to a type of question answering that goes beyond simply extracting information from a given text. Instead of pulling direct quotes or passages, it generates answers in a free-form manner, often rephrasing or summarizing the information to provide a more coherent and contextually relevant response. This approach allows for more flexibility and creativity in answering questions, as it can synthesize information from multiple sources or present it in a way that is not strictly tied to the original text.'

In [32]:
from langchain_core.runnables import RunnablePassthrough

In [33]:
rag_chain = (
    {"context": ensemble_retriever, "question": RunnablePassthrough()} |
    prompt |
    llm
)
rag_chain

{
  context: EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000023C24F23B00>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000023C2522D400>, k=3)], weights=[0.3, 0.7]),
  question: RunnablePassthrough()
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nUse the following pieces of context to answer the question at the end.\nIf you don\'t know the answer, just say that you do not have the relevant information needed to provide a verified answer, don\'t try to make up an answer.\nWhen providing an answer, aim for clarity and precision. Position yourself as a knowledgeable authority on the topic, but also be mindful to explain the information in a manner that is accessible and comprehensible to those without a technical background.\nAlways say "Do you have any more questions perta

In [34]:
query = "what is RAG token ?"

response = rag_chain.invoke(query)
response

AIMessage(content='RAG-Token, or Retrieval-Augmented Generation Token, is a model that combines retrieval and generation techniques to produce text. It operates by first retrieving a set of relevant documents based on a given input and then generating responses by leveraging the information from these documents. The model generates text in an autoregressive manner, meaning it predicts the next token (word or part of a word) based on the previously generated tokens and the retrieved documents.\n\nIn practical terms, RAG-Token retrieves the top K documents that are most relevant to the input and uses these documents to inform its generation process. It calculates a distribution for the next output token for each document and then combines these distributions to produce a final output. This allows RAG-Token to create more informed and contextually relevant responses compared to traditional generation models that rely solely on the input without external information.\n\nDo you have any mor

AIMessage(content='RAG-Token, or Retrieval-Augmented Generation Token, is a model that combines retrieval and generation techniques to produce text. It operates by first retrieving a set of relevant documents based on a given input and then generating text based on the information contained in those documents. \n\nThe process involves the following steps:\n\n1. **Document Retrieval**: The model retrieves the top K documents that are most relevant to the input query.\n2. **Token Generation**: For each token in the output, the model generates a probability distribution over possible next tokens based on the retrieved documents. This is done using an autoregressive approach, where the generation of each token depends on the previously generated tokens and the retrieved information.\n3. **Marginalization**: The model marginalizes over the retrieved documents to produce a final output token, effectively combining information from multiple sources.\n\nRAG-Token is particularly useful in tasks like question answering and text generation, where it can leverage external knowledge to enhance the quality and relevance of the generated text. It has been shown to outperform traditional models like BART in certain tasks, such as generating Jeopardy questions.\n\nDo you have any more questions pertaining to this instrument?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 237, 'prompt_tokens': 1961, 'total_tokens': 2198, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8bda4d3a2c', 'id': 'chatcmpl-CDwrhwocGTcrSHcAy8UqWuTjR4225', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--0412cff0-f254-45ec-9c79-325ad355974d-0', usage_metadata={'input_tokens': 1961, 'output_tokens': 237, 'total_tokens': 2198, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [35]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

In [36]:
# %pip install cohere

In [37]:
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

compressor = CohereRerank(cohere_api_key=COHERE_API_KEY, model="rerank-english-v3.0")

C:\Users\Bapan Bairagya\AppData\Local\Temp\ipykernel_22680\242068366.py:3: LangChainDeprecationWarning: The class `CohereRerank` was deprecated in LangChain 0.0.30 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-cohere package and should be used instead. To use it run `pip install -U :class:`~langchain-cohere` and import as `from :class:`~langchain_cohere import CohereRerank``.
  compressor = CohereRerank(cohere_api_key=COHERE_API_KEY, model="rerank-english-v3.0")


In [38]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=ensemble_retriever
)

In [39]:
query = "what is RAG token ?"

compressed_docs = compression_retriever.get_relevant_documents(query)
compressed_docs

C:\Users\Bapan Bairagya\AppData\Local\Temp\ipykernel_22680\1087271660.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  compressed_docs = compression_retriever.get_relevant_documents(query)


[Document(metadata={'page_label': '4', 'source': '../Retrieval-Augmented-Generation.pdf', 'title': '', 'page': 3, 'keywords': '', 'creator': 'LaTeX with hyperref', 'author': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'creationdate': '2021-04-13T00:48:38+00:00', 'total_pages': 19, 'subject': '', 'trapped': '/False', 'moddate': '2021-04-13T00:48:38+00:00', 'producer': 'pdfTeX-1.40.21', 'relevance_score': 0.99540824}, page_content='2.5 Decoding\nAt test time, RAG-Sequence and RAG-Token require different ways to approximatearg maxyp(y|x).\nRAG-Token The RAG-Token model can be seen as a standard, autoregressive seq2seq genera-\ntor with transition probability: p′\nθ(yi|x,y1:i−1) = ∑\nz∈top-k(p(·|x)) pη(zi|x)pθ(yi|x,zi,y1:i−1) To\ndecode, we can plug p′\nθ(yi|x,y1:i−1) into a standard beam decoder.\nRAG-Sequence For RAG-Sequence, the likelihood p(y|x) does not break into a conventional per-'),
 Document(metadata={'producer'

In [40]:
compressed_docs[0].page_content

'2.5 Decoding\nAt test time, RAG-Sequence and RAG-Token require different ways to approximatearg maxyp(y|x).\nRAG-Token The RAG-Token model can be seen as a standard, autoregressive seq2seq genera-\ntor with transition probability: p′\nθ(yi|x,y1:i−1) = ∑\nz∈top-k(p(·|x)) pη(zi|x)pθ(yi|x,zi,y1:i−1) To\ndecode, we can plug p′\nθ(yi|x,y1:i−1) into a standard beam decoder.\nRAG-Sequence For RAG-Sequence, the likelihood p(y|x) does not break into a conventional per-'

In [41]:
compressed_docs[0].metadata['relevance_score']

0.99540824

In [42]:
compressed_docs[1].metadata['relevance_score']

0.94796216

In [43]:
compressed_docs[2].metadata['relevance_score']

0.88602656

In [44]:
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=compression_retriever
)

hybrid_chain

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the user's question. \nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000023C28338A70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023C2833DF70>, root_client=<openai.OpenAI object at

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the user's question. \nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000240F686DA30>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000240F6866030>, root_client=<openai.OpenAI object at 0x00000240F36F0C20>, root_async_client=<openai.AsyncOpenAI object at 0x00000240F686DC10>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********')), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context'), retriever=ContextualCompressionRetriever(base_compressor=CohereRerank(client=<cohere.client.Client object at 0x00000240F60F88F0>, top_n=3, model='rerank-english-v3.0', cohere_api_key='cgxLYhsmPU0fqpvdB9KoBCYbIH4tF6eoqBVPS2D3', user_agent='langchain'), base_retriever=EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000240F3427350>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000240F58CB350>, k=3)], weights=[0.3, 0.7])))

In [45]:
response = hybrid_chain.invoke("What is Abstractive Question Answering?")
response

{'query': 'What is Abstractive Question Answering?',
 'result': 'Abstractive Question Answering is a type of question answering that goes beyond simply extracting answers from provided documents. Instead, it generates answers in a free-form manner, allowing for more flexibility and creativity in the responses. This approach can produce answers even when the correct information is not explicitly found in the retrieved documents, which is a significant advantage over extractive models that can only provide answers based on the text they have access to.'}

In [46]:
response['result']

'Abstractive Question Answering is a type of question answering that goes beyond simply extracting answers from provided documents. Instead, it generates answers in a free-form manner, allowing for more flexibility and creativity in the responses. This approach can produce answers even when the correct information is not explicitly found in the retrieved documents, which is a significant advantage over extractive models that can only provide answers based on the text they have access to.'

'Abstractive Question Answering is a type of question answering that goes beyond simply extracting answers from provided documents. Instead, it generates answers in a free-form manner, which means it can create responses that may not be directly found in the retrieved documents. This approach allows for more flexible and nuanced answers, as it can synthesize information and provide a coherent response based on the context of the question.'

In [47]:
query = "what is RAG token ?"

response_1 = hybrid_chain.invoke(query)
response_1

{'query': 'what is RAG token ?',
 'result': 'RAG-Token is a model that can be seen as a standard autoregressive sequence-to-sequence generator. It uses a transition probability to generate output tokens based on the input and previously generated tokens. Specifically, it approximates the likelihood of generating a token given the input and the previously generated tokens by considering a set of top-k candidates from a retrieval process. This model is designed to effectively generate responses by leveraging information from multiple documents.'}

In [48]:
response_1['result']

'RAG-Token is a model that can be seen as a standard autoregressive sequence-to-sequence generator. It uses a transition probability to generate output tokens based on the input and previously generated tokens. Specifically, it approximates the likelihood of generating a token given the input and the previously generated tokens by considering a set of top-k candidates from a retrieval process. This model is designed to effectively generate responses by leveraging information from multiple documents.'

'RAG-Token is a model that can be seen as a standard autoregressive sequence-to-sequence generator. It uses a transition probability to generate output tokens based on the input and previously generated tokens. Specifically, it approximates the likelihood of generating a token given the input and the previous tokens by considering a set of top-k candidates from a retrieval process. This model is designed to effectively generate responses by leveraging information from multiple documents during the decoding process.'

In [49]:
query = 'what is natural language processing ?'

response_2 = hybrid_chain.invoke(query)
response_2

{'query': 'what is natural language processing ?', 'result': "I don't know."}

In [50]:
query = 'what is natural language processing ?'

com_docs = compression_retriever.get_relevant_documents(query)

In [51]:
com_docs[0].metadata['relevance_score']

0.03149938

In [52]:
com_docs[1].metadata['relevance_score']

0.031439852

In [53]:
com_docs[2].metadata['relevance_score']

0.012479699